# 🧠 SynthDetect — Colab Training Workflow

This notebook trains the FAID contrastive encoder using real stylometric 
features extracted by the SynthDetect pipeline.

**Why Colab?** Neural network training is significantly faster on GPUs. 
**Safety Measure:** We mount Google Drive to immediately save checkpoints 
so you don't lose the model if Colab disconnects or the runtime expires.

### 📋 Instructions:
1. Select **Runtime > Change runtime type** and ensure **T4 GPU** (or better) is selected.
2. On the left side panel, click the Folder icon 📁 and upload your two feature files:
   - `all_features.npy`
   - `all_metadata.csv`
3. Click **Runtime > Run all**.
4. A prompt will appear asking to connect your Google Drive. Approve it.
5. When training completes, your models and FAISS index will be safely stored in your Drive!

## 1. Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn faiss-gpu

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
import faiss

## 2. Mount Google Drive Safety
This ensures that even if you close the tab, the trained weights are saved permanently in your Google Drive.

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a dedicated directory in your Drive for SynthDetect outputs
drive_save_dir = Path('/content/drive/MyDrive/SynthDetect_Outputs/')
drive_save_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Models will be safely saved to: {drive_save_dir}")

## 3. Data Loading
Load the pre-extracted stylometric features (`all_features.npy` and `all_metadata.csv`).

In [ ]:
# Verify files exist
features_path = Path('/content/all_features.npy')
metadata_path = Path('/content/all_metadata.csv')

if not features_path.exists() or not metadata_path.exists():
    raise FileNotFoundError("❌ Please upload all_features.npy and all_metadata.csv using the folder icon on the left panel!")

# Load Data
print("Loading real stylometric features...")
features = np.load(features_path).astype(np.float32)
metadata_df = pd.read_csv(metadata_path)

input_dim = features.shape[1]
print(f"Features shape: {features.shape} ({input_dim} linguistic dimensions)")

# We train the encoder to cluster by semantic 'source' (e.g., human vs chatgpt)
sources = metadata_df['source'].values
labels_str = sources.astype(str)

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels_str)
source_names = label_encoder.classes_

print("\nSource distribution in training data:")
print(metadata_df['source'].value_counts())

## 4. PyTorch Architecture Definition
We define the Multi-Layer Perceptron (MLP) and the Supervised Contrastive Loss function.

In [ ]:
class ContrastiveEncoder(nn.Module):
    """MLP contrastive encoder mapping raw features to stylometric embeddings."""
    def __init__(self, input_dim, hidden_dims, embedding_dim, dropout):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, embedding_dim))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        embedding = self.encoder(x)
        # L2 normalize embeddings for cosine similarity
        return torch.nn.functional.normalize(embedding, p=2, dim=1)

class SupConLoss(nn.Module):
    """Supervised Contrastive Loss (Khosla et al., 2020)"""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]
        similarity = torch.matmul(features, features.T) / self.temperature
        labels = labels.unsqueeze(1)
        mask = torch.eq(labels, labels.T).float().to(device)
        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=device)
        mask = mask * logits_mask
        exp_logits = torch.exp(similarity) * logits_mask
        log_prob = similarity - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-8)
        mask_sum = torch.clamp(mask.sum(dim=1), min=1)
        mean_log_prob = (mask * log_prob).sum(dim=1) / mask_sum
        return -mean_log_prob.mean()

class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

## 5. Training Hyperparameters Configuration

In [ ]:
CONFIG = {
    "input_dim": input_dim,
    "hidden_dims": [512, 256],
    "embedding_dim": 256,
    "dropout": 0.3,
    "temperature": 0.07,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,  # Real data needs more epochs
    "batch_size": 64,
    "early_stopping_patience": 10,
    "train_split": 0.8,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print(f"Training on device: {CONFIG['device'].upper()}")
if CONFIG['device'] != 'cuda':
    print("⚠️ WARNING: You are not using a GPU! Go to Runtime > Change runtime type and select T4 GPU.")

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# Prepare dataloaders
dataset = FeatureDataset(features, labels)
train_size = int(CONFIG["train_split"] * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(CONFIG["seed"])
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

## 6. Execute Training Loop

In [ ]:
model = ContrastiveEncoder(
    input_dim=CONFIG["input_dim"],
    hidden_dims=CONFIG["hidden_dims"],
    embedding_dim=CONFIG["embedding_dim"],
    dropout=CONFIG["dropout"],
).to(CONFIG["device"])

criterion = SupConLoss(temperature=CONFIG["temperature"])
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

best_val_loss = float("inf")
patience_counter = 0
best_model_state = None
train_losses = []
val_losses = []

print(f"\n🚀 Starting Training on {len(train_dataset)} samples...")

for epoch in range(1, CONFIG["epochs"] + 1):
    # Train pass
    model.train()
    epoch_train_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(CONFIG["device"])
        batch_labels = batch_labels.to(CONFIG["device"])

        optimizer.zero_grad()
        embeddings = model(batch_features)
        loss = criterion(embeddings, batch_labels)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation pass
    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for batch_features, batch_labels in val_loader:
            batch_features = batch_features.to(CONFIG["device"])
            batch_labels = batch_labels.to(CONFIG["device"])
            embeddings = model(batch_features)
            loss = criterion(embeddings, batch_labels)
            epoch_val_loss += loss.item()

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    scheduler.step()

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{CONFIG['epochs']} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    if avg_val_loss < best_val_loss - 0.001:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stopping_patience"]:
            print(f"⏹️ Early stopping triggered at epoch {epoch}")
            break

print(f"✅ Training completed! Best Val Loss: {best_val_loss:.4f}")

## 7. Automatic Save to Google Drive
We now backup the best weights directly to your Google Drive to prevent data loss.

In [ ]:
model.load_state_dict(best_model_state)

checkpoint_path = drive_save_dir / f"contrastive_encoder_v1_{datetime.now().strftime('%Y%m%d_%H%M')}.pth"

torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "source_names": list(source_names),
}, str(checkpoint_path))

print(f"💾 Permanently Saved Model Checkpoint to Google Drive at:")
print(f"   -> {checkpoint_path}")

## 8. Evaluation Visualization (t-SNE)
Let's visualize how well the network separated human vs AI writing styles.

In [ ]:
model.eval()
all_embeddings = []
all_labels_list = []

with torch.no_grad():
    for batch_features, batch_labels in val_loader:
        batch_features = batch_features.to(CONFIG["device"])
        embeddings = model(batch_features)
        all_embeddings.append(embeddings.cpu().numpy())
        all_labels_list.append(batch_labels.numpy())

all_embeddings = np.vstack(all_embeddings)
all_labels_arr = np.concatenate(all_labels_list)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(all_embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette("husl", len(source_names))

for idx, source in enumerate(source_names):
    mask = all_labels_arr == idx
    ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
               c=[colors[idx]], label=source, alpha=0.7, s=30)

ax.set_title("t-SNE Visualization of Contrastive Stylometric Embeddings")
ax.legend()
plt.tight_layout()

tsne_path = drive_save_dir / "tsne_visualization.png"
plt.savefig(tsne_path, dpi=150)
print(f"📈 Saved t-SNE plot to {tsne_path}")
plt.show()

## 9. Build and Save FAISS Index
The vector database needs to be built with your newly trained encoder and then saved to Drive.

In [ ]:
print("Building FAISS Index database for k-NN attribution...")

# Encode all training data
train_embeddings = []
model.eval()
with torch.no_grad():
    for batch_features, _ in train_loader:
        batch_features = batch_features.to(CONFIG["device"])
        embeddings = model(batch_features)
        train_embeddings.append(embeddings.cpu().numpy())

train_embeddings = np.vstack(train_embeddings).astype(np.float32)
faiss.normalize_L2(train_embeddings)

index = faiss.IndexFlatIP(CONFIG["embedding_dim"])

# If using GPU and faiss-gpu:
if CONFIG['device'] == 'cuda':
    res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)

index.add(train_embeddings)

# Transfer back to CPU for saving
if CONFIG['device'] == 'cuda':
    index = faiss.index_gpu_to_cpu(index)

index_path = drive_save_dir / "main_index.faiss"
faiss.write_index(index, str(index_path))

# Also extract metadata associated with the encoded items
metadata = [
    {
        "source": source_names[int(label)],
        "label": "human" if "human" in source_names[int(label)] else "ai"
    }
    for label in (batch_labels for _, batch_labels in train_loader)
]

# We need the full flattened list for metadata
metadata_flat = []
for batch_features, batch_labels in train_loader:
    for lbl in batch_labels:
        source_name = source_names[int(lbl.item())]
        metadata_flat.append({
            "source": source_name,
            "label": "human" if "human" == source_name else "ai"
        })

meta_path = drive_save_dir / "main_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata_flat, f)

print(f"✅ FAISS Data successfully built and securely saved to Drive:")
print(f"   -> {index_path}")
print(f"   -> {meta_path}")

🎉 **YOU'RE ALL SET!**

Please download these 3 files from your Google Drive `SynthDetect_Outputs` folder:
1. `contrastive_encoder_v1_...pth`
2. `main_index.faiss`
3. `main_metadata.json`

And place them locally into your project inside `data/models/` and `data/vector_db/` respectively.